In [1]:
#|default_exp models

In [2]:
#| export
from pathlib import Path
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv,LayerNorm,Sequential
from torch import autocast

In [3]:
#| export
def good_luck():
    return True

In [4]:
#| export
class MSLA(nn.Module):
    def __init__(self, dim, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.eps=1.0e-15
        
        self.qkv = nn.Linear(dim, dim*3, bias=True)
        self.proj = nn.Linear(dim, dim)
       
    @autocast(enabled=False, device_type="cuda")
    def forward(self, x):
        N,C = x.shape
        qkv = self.qkv(x.float())
        qkv = qkv.view(N, 3, self.num_heads, -1).permute(1,2,0,3)
        q, k, v = qkv[0], qkv[1], qkv[2] # h N c
        
        q = F.elu(q) + 1.0
        k = F.elu(k) + 1.0
        v = F.pad(v, (0, 1), mode="constant", value=1.0)
        
        kv = torch.matmul(k.transpose(-1, -2), v)
        out = torch.matmul(q, kv)
        out = out[..., :-1] / (out[..., -1:] + self.eps)
        out = out.permute(1,0,2).reshape(N,C)
        out = self.proj(out)
        
        return out
    
class GraphResABlock(nn.Module):
    def __init__(self, in_features, num_heads=8, act_layer=nn.GELU, drop=0.0):
        super().__init__()
        self.norm1 = LayerNorm(in_features, mode='node')
        self.attn = MSLA(in_features, num_heads)
        
        self.norm2 = LayerNorm(in_features, mode='node')
        self.act = act_layer()
        self.conv = SAGEConv(in_features, in_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x, edge_index):
        x = x + self.attn(self.norm1(x))
        x = x + self.drop(self.conv(self.act(self.norm2(x)),edge_index))
        return x
    
    
class GraphModel(nn.Module):
    def __init__(self, nh=256, depth=8, nout=None, ps=0, use_checkpoint=True):
        super().__init__()
        in_features = 140
        opcode_n = 120
        if nout is None: nout = nh
        self.use_checkpoint = use_checkpoint
        
        self.emb_opcode = nn.Embedding(opcode_n, nh)
        self.emb_config = nn.Embedding(2, nh)
        self.proj_in = nn.Linear(in_features, nh)
        self.blocks = nn.ModuleList([GraphResABlock(nh, drop=ps) 
                                     for _ in range(depth)])
        self.proj_out = nn.Linear(nh, nout)
        
    def forward(self, data):
        #print(data['node_opcode'].shape, data['node_config_ids'].shape)
        
        x_feat = data['node_feat'].squeeze(0) #add config features
        x_opcode = data['node_opcode'].squeeze(0)
        edge_index = data['edge_index'].squeeze(0).T
        config_ids = data['node_config_ids'].squeeze(0)
        x_config = torch.zeros_like(x_opcode)
        x_config[config_ids] = 1
        
        x = self.proj_in(x_feat) + self.emb_opcode(x_opcode) + self.emb_config(x_config)
        for blk in self.blocks:
                x = blk(x,edge_index)
        x = self.proj_out(x)
        x = x[config_ids]
        
        return x

class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1)*p)
        self.eps = eps

    def forward(self, x):
        return self.gem(x, p=self.p, eps=self.eps)
        
    def gem(self, x, p=3, eps=1e-6):
        return (x.clamp(min=eps).pow(p)).mean(-2).pow(1./p)
        
    def __repr__(self):
        return self.__class__.__name__ + '(' + 'p=' + '{:.4f}'.format(self.p.data.tolist()[0]) + \
                ', ' + 'eps=' + str(self.eps) + ')'
    
    
    
class ConcatPool(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.gem = GeM(p=p,eps=eps)
    
    def forward(self, x):
        return torch.cat([x.mean(-2),self.gem(x)],-1)
    
class TPUModelV0(nn.Module):
    def __init__(self, nh=256, depth=12, psg=0.1, ps=0.1):
        super().__init__()
        n_cemb = 4
        self.gnn = GraphModel(nh=nh,depth=depth,ps=psg)
        self.config_emb = nn.Embedding(7,n_cemb)
        self.head = nn.Sequential(
                nn.Linear(nh+18*n_cemb,nh),nn.LayerNorm(nh),
                nn.GELU(),nn.Dropout(ps),
                nn.Linear(nh,nh),nn.LayerNorm(nh),
                nn.GELU(),nn.Dropout(ps),
                ConcatPool(),
                nn.Linear(2*nh,nh),nn.LayerNorm(nh),
                nn.GELU(),nn.Dropout(ps),
                nn.Linear(nh,nh),nn.LayerNorm(nh),
                nn.GELU(),nn.Dropout(ps),nn.Linear(nh,1))
        
    def forward(self, data):
        emb_g = self.gnn(data)
        config_feat = data['node_config_feat'].squeeze(0)
        config_feat = self.config_emb(config_feat).flatten(-2)
        B = config_feat.shape[0]
        x = torch.cat([emb_g.unsqueeze(0).repeat(B,1,1),config_feat],-1)
        
        if self.training:
            p = self.head(x)
        else:
            n = max(1, B // 128)
            ps = []
            for xi in x.chunk(n):
                p = self.head(xi)
                ps.append(p)
            p = torch.cat(ps,0)

        return {'pred':p.view(1,-1)}

In [5]:
class CONFIG:
    path = Path('/opt/slh/tpuoptimize/data/layout_converted_z2_1000')
    nmax_t=256
    nmax_v=-1
    model_kwargs = {"nh":256, "depth":12, "psg":0.1, "ps":0.1}
    

In [6]:
data = torch.load('batch.pt')[0]


In [8]:
model = TPUModelV0(**CONFIG.model_kwargs).eval()
with torch.no_grad():
        out = model(data)

In [9]:
data['node_config_feat'].shape

torch.Size([1, 256, 72, 18])

In [10]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()